# Calculate neuron "f-factor"

Copyright (c) 2026 Open Brain Institute

Authors: Michael W. Reimann

last modified: 06.2026

### Important: Morphology with spines only!
**Please note that this notebook only works with CellMorphologies that have reconstructed spines** 
You can create such morphologies by running our "EM Cell Mesh Skeletonization" workflow!

## Summary
We calculate how spiny a neuron is.
That is, by how much does the surface area of each part of the neuron increase through spines?

## Imports and select project

Select the project you want to work with. 
Important: Selection of the project determines which neuron morphologies you have access to!

In [ ]:
from morph_spines import load_morphology_with_spines

import obi_auth
import pylmesh
import pandas
import numpy as np

from entitysdk import Client, types, models
from obi_one import CellMorphologyFromID
from entitysdk.models import EMCellMesh, EMDenseReconstructionDataset, CellMorphology
from obi_notebook. get_projects import get_projects
from obi_notebook.get_environment import get_environment
from ipywidgets import widgets

import logging
loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    logger.setLevel(logging.ERROR)

env_ = get_environment()
token = obi_auth.get_token(environment=env_, auth_mode="daf")
project_context = get_projects(token, env=env_)

## Select EM Dataset to consider
Skeletonized morphologies stem from an electron microscopy dataset.

The following dropdown displays all EM datasets accessible on the OBI platform. Select the one to consider.

**NOTE**: 
For some of the datasets you may not have access to any skeletonized morphologies. In that case, try a different one or run the `Skeletonization` workflow on the main platform.

In [ ]:
client = Client(project_context=project_context, token_manager=token, environment=env_)

em_datasets = client.search_entity(entity_type=models.EMDenseReconstructionDataset).all()

sel_em = widgets.Dropdown(options={dataset.name: dataset for dataset in em_datasets})
display(sel_em)


## Select neuron with spines

Below, you can directly paste the ID of a neuron morphology that you have skeletonized.
If you do so, please ensure that you have selected the correct **Project** above, otherwise you will get a 404 error.

Alternatively, you will be presented with a list of neuron morphologies with spines that you have access to.

In [ ]:
neuron_id = "PASTE ID IN HERE"

sel_nrn = None
if neuron_id == "PASTE ID IN HERE":
    # Find all CellMorphologies from MICrONS
    derivations = client.search_entity(entity_type=models.Derivation, query={
        "derivation_type": types.DerivationType.em_dense_reconstruction_dataset_cell_morphology,
        "used__id": sel_em.value.id
    })
    morphologies = [client.get_entity(entity_id=derivation.generated.id, entity_type=CellMorphology)
                    for derivation in derivations]
    sel_nrn = widgets.Dropdown(options={m.description: m for m in morphologies})
    display(sel_nrn)



## Download the data into the notebook

Here, we access the data from the database and load it.
This can take a few seconds.

In [ ]:
if sel_nrn is not None:
    morphology = CellMorphologyFromID(id_str=str(sel_nrn.value.id))
else:
    morphology = CellMorphologyFromID(id_str=neuron_id)
# Where to place the neuron and mesh
mesh_path = "neuron_mesh.glb"
neuron_path = "neuron_with_spines.h5"

# Load spiny neuron
morphology.write_spiny_neuron_h5(path_to=neuron_path, db_client=client)
m = load_morphology_with_spines(neuron_path, load_meshes=True)
print(f"Spine count: {m.spines.spine_count}")

# Download and load mesh
mesh = morphology.source_mesh_entity(db_client=client)
client.download_file(entity_id=mesh.id, entity_type=models.EMCellMesh, asset_id=mesh.assets[0].id, output_path=mesh_path)
neuron_mesh = pylmesh.load_mesh(mesh_path)

## Calculate surface areas
Here, we calculate the total surface area of each part of the morphology.

  - pt_df: A DataFrame with one entry for each reconstructed *segment* of the morphology. The segment is identified by its *section* and *segment* id. The column "area" is its surface area.
  - spine_df: A DataFrame with one entry for each *spine*. It lists separately its *head* and *neck* surface area. It also identifies for each spine the *section* and *segment* it is rooted on. This allows us to combine the DataFrame with pt_df.

In [ ]:
import f_factor_helpers

vertices, faces = f_factor_helpers.mesh_vertices_and_faces(neuron_mesh)
pt_df = f_factor_helpers.morph_segments_dataframe(m.morphology)

f_factor_helpers.add_mesh_area(pt_df, vertices, faces, column_name="all_surface_area")
soma_vertices = m.soma.soma_mesh_points
soma_faces = m.soma.soma_mesh_triangles
f_factor_helpers.add_mesh_area(pt_df, soma_vertices, soma_faces, column_name="soma_surface_area")
pt_df["area"] = pt_df["all_surface_area"] - pt_df["soma_surface_area"]

spine_df = f_factor_helpers.spine_area_df(m)

# Calculate per-section averages

Here, we calculate total areas per *section*. And then calculate the values we are interested in. 

Estimates can be noisy for very short sections, so we replace their values with a value interpolated based on soma path distance.

In [ ]:
counts = spine_df.groupby("afferent_section_id")["neck_area"].count()
counts.name = "spine_count"
master_df = pandas.concat([
    spine_df.groupby("afferent_section_id")[["neck_area", "head_area"]].sum(),
    counts,
    pt_df.groupby("afferent_section_id")[["area", "length"]].sum()
], axis=1).fillna(0).sort_index()

master_df["spine_density"] = master_df["spine_count"] / master_df["length"]
master_df["smooth_area"] = master_df["area"] - master_df["neck_area"] - master_df["head_area"]

master_df["f_factor"] = master_df["area"] / master_df["smooth_area"]
master_df["f_factor_neck"] = (master_df["smooth_area"] + master_df["neck_area"]) / master_df["smooth_area"]
master_df["f_factor_head"] = (master_df["smooth_area"] + master_df["head_area"]) / master_df["smooth_area"]
master_df["head_neck_ratio"] = (master_df["head_area"] / master_df["neck_area"]).fillna(1.0)

sec_tip_soma_dist = pt_df.groupby("afferent_section_id")["soma_distance"].max()
master_df["soma_distance"] = sec_tip_soma_dist[master_df.index]

f_factor_helpers.interpolate_for_short_secs(m.morphology, master_df, min_len=5.0)
master_df

# Plot
We plot the results.

Each black dot represents a *section*. It is placed at the soma path distance of its tip (most distal point). Its *size* represents the length of the section.

Each colored line represent a neurite. The *axon* is included! We present smoothed values for the neurite.

You can select the amount of smoothing with the slider.

The measure to be plotted is selected from the dropdown.

In [ ]:
from matplotlib import pyplot as plt

select_col = widgets.Dropdown(options=f_factor_helpers.Y_OPTIONS)
select_smooth = widgets.FloatSlider(value=15.0, min=1.0, max=50.0, step=1.0)

def plot_for_col(col_plot, smooth):
    plt.figure(figsize=(6.5, 3.5))
    for nrt_id in range(len(m.morphology.neurites)):
        nrt = f_factor_helpers.downline_of(m.morphology.neurites[nrt_id].root_node)
        x, y = f_factor_helpers.smoothed(master_df.loc[nrt], "soma_distance", col_plot, smooth)
        plt.plot(x, y, label=f"Neurite #{nrt_id}")

    plt.scatter(master_df["soma_distance"], master_df[col_plot], s=master_df["length"]/2., color="black", alpha=0.5)
    plt.legend()
    plt.gca().set_xlabel("Soma path distance (um)")
    plt.gca().set_ylabel(col_plot)
    plt.gca().set_frame_on(False)

display(widgets.interactive(plot_for_col, col_plot=select_col, smooth=select_smooth))

# Visualize
We visualize the results also as a colored 3d plot of the surface mesh.

We drastically simplify the surface mesh for speed of presentation. 
It can still take up to a minute to display, so please remain patient!

Simplify mesh

In [ ]:
!pip install fast-simplification
simpl_vertices, simpl_faces = f_factor_helpers.mesh_vertices_and_faces(neuron_mesh, 0.025)

In [ ]:
from matplotlib import cm
from scipy.spatial import KDTree
import k3d

tree = KDTree(pt_df[["x", "y", "z"]].to_numpy())
_, v_pt_idx = tree.query(simpl_vertices)

vtx_y_vals = master_df.loc[pt_df.loc[v_pt_idx, "afferent_section_id"], select_col.value]
y_min = 0.0#f_factor_helpers.Y_MINIMA[select_col.value]
y_max = np.percentile(vtx_y_vals, 95)
vtx_y_norm = np.minimum(np.maximum((vtx_y_vals - y_min) / (y_max - y_min), 0.0), 1.0)

vtx_cols = f_factor_helpers.pack_rgb((cm.brg(vtx_y_norm) * 255).astype(np.uint8))

simpl_vertices = np.array(simpl_vertices).astype(np.float32)
simpl_faces = np.array(simpl_faces).astype(np.float32)

plot_face = k3d.plot(background_color=0x000000, grid_visible=False)
plot_face += k3d.mesh(
    simpl_vertices, simpl_faces,
    colors=vtx_cols,
    color_map=[],   # disable built-in colormap
    attribute=[],
)
plot_face.display()